# HGNN — Entrenamiento corregido (Outfit Compatibility) — versión PyTorch

Reescritura en **PyTorch** de la versión anterior (TensorFlow), por los problemas que
comentas con la GPU en las últimas versiones de TF. La arquitectura y la lógica son
exactamente las mismas — solo cambia el framework — más barras de progreso con
**tqdm** para ver el avance dentro de cada epoch (no solo al final).

## Recordatorio de las correcciones respecto a tu notebook original

1. **La propagación sobre el hipergrafo ahora usa `Theta` de verdad** —
   `X' = ReLU(Linear(Theta @ X))` sobre una matriz de nodos-categoría construida
   (`scatter`) y recuperada (`gather`) por outfit, en vez de un MLP por-ítem que
   ignoraba el grafo.
2. **Esquema multimodal — extensión deliberada, no algo que describa el paper de
   HGNN.** Verificamos contra el paper original (Li et al. 2022, OCPHN) y contra tu
   paper subido: **la implementación de HGNN que describen es solo-visual**
   (`"we only used the image information of the items in this paper"`, Li et al.
   2022). La mención de `β`/multimodal en la sección de HGNN de tu paper es casi con
   certeza un copy-paste del párrafo de NGNN, no una descripción real de esa parte.
   Decidiste añadir multimodal de todas formas porque la Tabla 2 de tu paper muestra
   que ayuda en **NGNN** — es una extrapolación razonable, pero una hipótesis tuya a
   validar empíricamente con tus propios resultados, no una réplica. El mecanismo de
   fusión (`x_s = β·x_vis + (1-β)·x_txt`, Ec. 13, + pérdida de consistencia `L_con`,
   Ec. 14) está tomado prestado de NGNN, aplicado aquí a HGNN por analogía.
3. **Score de compatibilidad fiel a la Ec. 12**:
   `x_s = Σ_i sigmoid(θ(h_i)) · leaky_relu(δ(h_i))`.
4. **Split**: `nondisjoint` por defecto (benchmark estándar, 53 306 outfits de train).
5. **Filtrado de categorías/outfits fiel al paper**: categorías con >100 apariciones
   en train, outfits con ≥3 ítems válidos.
6. **Pérdida**: BPR por defecto (`-log sigmoid(x_pos - x_neg)`), con triplet-con-margen
   disponible como alternativa.
7. **Regularización**: L2 vía `weight_decay` del optimizador Adam (equivalente
   práctico al `kernel_regularizer` de Keras) + dropout opcional (no está en el paper).
8. **Checkpointing + early stopping** sobre AUC de validación.

## Nota sobre `weight_decay` en PyTorch vs `kernel_regularizer` en TF

En la versión de TensorFlow, la L2 se aplicaba solo a los pesos (`kernel`) de cada
capa `Dense`, no a los sesgos (`bias`). El optimizador se construye con dos grupos de
parámetros (`decay_params`/`no_decay_params`, sección 5) para replicar exactamente
ese comportamiento: `weight_decay` solo se aplica a matrices de pesos (`ndim>=2`), no
a sesgos ni a los parámetros de `LayerNorm`. Esto no es solo fidelidad al paper —
aplicar `weight_decay` a los sesgos por igual facilita que el modelo colapse a una
salida constante para minimizar `L_con` de forma trivial (ver nota en
`CONFIG["lambda_con"]`).


## 0. Configuración

In [ ]:
import os
import json
import random

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

POLYVORE_PATH = "./polyvore_kaggle/polyvore_outfits/"

PATHS = {
    "bow_dir":       POLYVORE_PATH + "data/preprocessed/BoW_for_HGNN/",
    "img_dir":       POLYVORE_PATH + "data/preprocessed/InceptionV3_for_HGNN/",
    "text_features": POLYVORE_PATH + "data/processed/text_features.csv",
    "checkpoints":   POLYVORE_PATH + "checkpoints_HGNN/",
}
os.makedirs(PATHS["checkpoints"], exist_ok=True)

CONFIG = {
    # --- datos ---
    "split": "nondisjoint",        # "nondisjoint" (benchmark estandar, 53306 train) o "disjoint" (16995 train)
    "min_category_freq": 100,      # categorias con < 100 apariciones en train se descartan (paper, sec 1.1.2)
    "min_outfit_items": 3,         # outfits con < 3 items validos se descartan (paper, sec 1.1.2)

    # --- arquitectura (valores confirmados por el codigo oficial de OCPHN, no por el
    #     re-derivado del paper subido) ---
    "hidden_dim": 16,              # d. El codigo oficial usa 16, no 12 (el 12 era el optimo re-derivado
                                    # del paper subido, con un dataset mas pequeno)
    "n_layers": 1,                 # T. CRITICO: el codigo oficial usa n_steps=1, no 3. Con Theta bastante
                                    # densa (~90% no-cero) y T=3, es un caso de libro de "oversmoothing":
                                    # tras varias rondas de mezcla, las representaciones de items distintos
                                    # convergen a un promedio parecido y pierden la identidad especifica que
                                    # hace falta para diferenciar outfits -- coincide con rank_loss clavada
                                    # en ln(2) incluso sobreajustando 200 outfits con lr alto.
    "beta": 0.2,                   # peso del canal visual en la fusion tardia (Ec. 13)
    "l2_reg": 0.001,               # weight_decay del optimizador. El codigo oficial usa 0.1 (mucho mas fuerte,
                                    # y ademas promediado sobre todos los tensores, no sumado) -- si tras bajar
                                    # n_layers a 1 sigue sin converger bien, prueba a subir esto tambien
    "dropout_rate": 0.1,           # NO esta en el paper, se anade como regularizacion extra. 0.0 para desactivar

    # --- perdida ---
    "loss_type": "bpr",            # "bpr" (-log sigmoid(diff), estandar), "triplet" (margen explicito) o
                                    # "paper_sigmoid" (-sigmoid(diff), sin log -- la que usa el codigo oficial;
                                    # gradiente maximo en diff=0 y se desvanece en ambas direcciones, distinto
                                    # comportamiento de optimizacion que BPR estandar)
    "triplet_margin": 0.2,         # solo se usa si loss_type == "triplet"
    "lambda_con": 0.01,             # peso de la perdida de consistencia visual-textual (Ec. 14). No especificado
                                     # en el paper. OJO: valores altos (p.ej. 0.1) pueden inducir un colapso de
                                     # representacion -- el modelo hace r_vis=r_txt=0 para satisfacer L_con
                                     # trivialmente en vez de resolver el ranking. Si ves "con" cayendo a 0 exacto
                                     # mientras "rank" se queda clavado en ln(2)=0.693, baja esto aun mas (o a 0
                                     # temporalmente para confirmar el diagnostico).

    # --- entrenamiento ---
    "batch_size": 16,              # paper y codigo oficial coinciden
    "learning_rate": 0.0001,       # el codigo oficial usa 0.0001, no 0.001 (tambien confirmado por el propio
                                    # grid search del paper: "optimal when the learning rate was 0.0001")
    "max_epochs": 30,
    "early_stopping_patience": 5,
    "eval_n_samples_per_epoch": 800,   # subset de validacion evaluado cada epoch (evaluacion completa es lenta)
    "num_workers": 0,              # sube esto en Linux si quieres paralelizar la carga de batches
}

print(json.dumps(CONFIG, indent=2))


## 1. Cargar splits, embeddings precomputados y metadata de categorías

### 1.1 Splits (train/valid/test)


In [ ]:
split_dir = POLYVORE_PATH + CONFIG["split"] + "/"

with open(split_dir + "train.json") as f:
    train_outfits_raw = json.load(f)
with open(split_dir + "valid.json") as f:
    valid_outfits_raw = json.load(f)
with open(split_dir + "test.json") as f:
    test_outfits_raw = json.load(f)

print(f"Split usado: {CONFIG['split']}")
print(f"  train: {len(train_outfits_raw)} outfits")
print(f"  valid: {len(valid_outfits_raw)} outfits")
print(f"  test:  {len(test_outfits_raw)} outfits")


### 1.2 Embeddings precomputados (BoW + InceptionV3) del notebook anterior

In [ ]:
# --- Visual (InceptionV3, denso) ---
img_embeddings = np.load(PATHS["img_dir"] + "image_embeddings.npy")
img_item_ids   = np.load(PATHS["img_dir"] + "item_ids.npy", allow_pickle=True).astype(str)
img_id_to_row  = {iid: i for i, iid in enumerate(img_item_ids)}
VIS_DIM = img_embeddings.shape[1]

# --- Textual (BoW, sparse) ---
bow_matrix   = sp.load_npz(PATHS["bow_dir"] + "bow_features.npz")
bow_item_ids = np.load(PATHS["bow_dir"] + "item_ids.npy", allow_pickle=True).astype(str)
bow_id_to_row = {iid: i for i, iid in enumerate(bow_item_ids)}
TXT_DIM = bow_matrix.shape[1]

print(f"Visual (InceptionV3): {img_embeddings.shape}  (dim={VIS_DIM})")
print(f"Textual (BoW):        {bow_matrix.shape}  (dim={TXT_DIM})")

def get_embeddings(item_id):
    """Devuelve (vis_vec, txt_vec) para un item_id, o None si falta en alguna modalidad."""
    ir = img_id_to_row.get(item_id)
    br = bow_id_to_row.get(item_id)
    if ir is None or br is None:
        return None
    vis = img_embeddings[ir]
    txt = np.asarray(bow_matrix[br].todense()).ravel().astype(np.float32)
    return vis, txt


### 1.2.1 Diagnóstico: ¿tienen señal explotable estos embeddings?

Antes de seguir revisando el modelo, comprobamos la salud de los datos en sí. Si una
fracción grande de `image_embeddings.npy` son vectores cero (el *fallback* para ítems
sin imagen encontrada en notebook 1) o si los embeddings son casi idénticos entre
ítems distintos, ningún modelo -por bien construido que esté- va a poder aprender
compatibilidad a partir de ellos.


In [ ]:
# % de vectores exactamente cero (posible fallback de imagen/BoW faltante)
vis_norms = np.linalg.norm(img_embeddings, axis=1)
n_vis_zero = int((vis_norms < 1e-8).sum())
print(f"Visual: {n_vis_zero}/{len(img_embeddings)} vectores exactamente cero ({n_vis_zero/len(img_embeddings):.2%})")
print(f"Visual: norma media={vis_norms.mean():.4f}  std={vis_norms.std():.4f}  min={vis_norms.min():.4f}  max={vis_norms.max():.4f}")

bow_row_sums = np.asarray(bow_matrix.sum(axis=1)).ravel()
n_txt_zero = int((bow_row_sums < 1e-8).sum())
print(f"\nTextual: {n_txt_zero}/{bow_matrix.shape[0]} vectores exactamente cero ({n_txt_zero/bow_matrix.shape[0]:.2%})")
print(f"Textual: palabras activas por item -- media={bow_row_sums.mean():.2f}  std={bow_row_sums.std():.2f}")

# Similitud coseno entre pares de items al azar -- si es sistematicamente muy alta (>0.95),
# los embeddings visuales podrian no ser lo bastante discriminativos entre items distintos.
rng_check = np.random.RandomState(SEED)
idx_sample = rng_check.choice(len(img_embeddings), size=min(500, len(img_embeddings)), replace=False)
sample = img_embeddings[idx_sample]
sample_norm = sample / (np.linalg.norm(sample, axis=1, keepdims=True) + 1e-8)
cos_sim = sample_norm @ sample_norm.T
np.fill_diagonal(cos_sim, np.nan)
print(f"\nSimilitud coseno entre 500 items al azar (visual):")
print(f"  media={np.nanmean(cos_sim):.4f}  std={np.nanstd(cos_sim):.4f}  max={np.nanmax(cos_sim):.4f}")
print(f"  (si la media es muy alta, p.ej. >0.9, los embeddings apenas distinguen items entre si)")


### 1.3 Categorías por ítem y filtrado (fiel al paper, sec. 1.1.2)

Se descartan categorías con menos de `CONFIG["min_category_freq"]` apariciones en el
split de **entrenamiento**, y luego se descartan de cada outfit los ítems cuya
categoría fue eliminada. Los outfits que se quedan con menos de
`CONFIG["min_outfit_items"]` ítems válidos se descartan por completo.


In [ ]:
df_text = pd.read_csv(PATHS["text_features"], dtype={'item_id': str})
item_to_category = dict(zip(df_text['item_id'], df_text['sub_category']))

# Frecuencia de categoria SOLO en train (asi el hipergrafo se construye igual que en el paper)
from collections import Counter
cat_freq_train = Counter()
for outfit in train_outfits_raw:
    for item in outfit['items']:
        cat = item_to_category.get(str(item['item_id']))
        if cat:
            cat_freq_train[cat] += 1

valid_categories = sorted([c for c, n in cat_freq_train.items() if n >= CONFIG["min_category_freq"]])
category_to_idx  = {c: i for i, c in enumerate(valid_categories)}
N_CATEGORIES = len(category_to_idx)

print(f"Categorias antes del filtro: {len(cat_freq_train)}")
print(f"Categorias tras filtro (>= {CONFIG['min_category_freq']} en train): {N_CATEGORIES}")


def filter_outfits(outfits_raw):
    """Se queda solo con items validos (categoria valida + embeddings disponibles)
    y descarta outfits con menos de min_outfit_items resultantes."""
    filtered = []
    for outfit in outfits_raw:
        valid_items = []
        for item in outfit['items']:
            item_id = str(item['item_id'])
            cat = item_to_category.get(item_id)
            if cat in category_to_idx and get_embeddings(item_id) is not None:
                valid_items.append({'item_id': item_id, 'category': cat})
        if len(valid_items) >= CONFIG["min_outfit_items"]:
            filtered.append({'items': valid_items})
    return filtered

train_outfits = filter_outfits(train_outfits_raw)
valid_outfits = filter_outfits(valid_outfits_raw)
test_outfits  = filter_outfits(test_outfits_raw)

print(f"\nOutfits validos tras filtrado:")
print(f"  train: {len(train_outfits)} / {len(train_outfits_raw)}")
print(f"  valid: {len(valid_outfits)} / {len(valid_outfits_raw)}")
print(f"  test:  {len(test_outfits)} / {len(test_outfits_raw)}")

# Lista de items validos (para el muestreo de negativos) = union de items validos de TODOS los splits
all_valid_items = set()
for split in (train_outfits, valid_outfits, test_outfits):
    for outfit in split:
        for item in outfit['items']:
            all_valid_items.add(item['item_id'])
all_valid_items = list(all_valid_items)
print(f"\nItems validos disponibles para muestreo negativo: {len(all_valid_items)}")


## 2. Construcción del hipergrafo (H, W, D_v, D_e, Θ)

Igual que antes: se construye con `scipy.sparse` (no depende del framework de deep
learning) y al final se convierte a un tensor denso de PyTorch — `Θ` es pequeña
(`N_CATEGORIAS × N_CATEGORIAS`, típicamente <150×150), así que no merece la pena
mantenerla sparse durante el entrenamiento.


In [ ]:
def build_fashion_hypergraph(outfits, category_to_idx):
    n_categories = len(category_to_idx)
    n_outfits    = len(outfits)

    rows, cols, outfit_weights = [], [], []
    for outfit_idx, outfit in enumerate(outfits):
        outfit_categories = {item['category'] for item in outfit['items']}
        for category in outfit_categories:
            rows.append(category_to_idx[category])
            cols.append(outfit_idx)
        outfit_weights.append(len(outfit_categories))

    data = np.ones(len(rows), dtype=np.float64)
    H = sp.csr_matrix((data, (rows, cols)), shape=(n_categories, n_outfits))
    W = sp.diags(outfit_weights)

    D_v = np.array(H.sum(axis=1)).flatten()
    D_e = np.array(H.sum(axis=0)).flatten()

    print(f"Matriz de incidencia H: {H.shape}  -- densidad: {H.nnz / (n_categories*n_outfits):.4f}")
    return H, W, D_v, D_e


def compute_hypergraph_laplacian(H, W, D_v, D_e):
    """Theta = D_v^-1/2 . H . W . D_e^-1 . H^T . D_v^-1/2  (Li et al. 2022)"""
    D_v_inv_sqrt = sp.diags(np.where(D_v > 0, D_v ** -0.5, 0))
    D_e_inv      = sp.diags(np.where(D_e > 0, D_e ** -1.0, 0))
    Theta = D_v_inv_sqrt @ H @ W @ D_e_inv @ H.T @ D_v_inv_sqrt
    return Theta


H_train, W_train, D_v, D_e = build_fashion_hypergraph(train_outfits, category_to_idx)
Theta_sparse = compute_hypergraph_laplacian(H_train, W_train, D_v, D_e)

Theta = torch.tensor(Theta_sparse.toarray(), dtype=torch.float32, device=device)
print(f"Theta: {tuple(Theta.shape)}")


### 2.1 Diagnóstico: ¿es `Theta` degenerada?

Antes de seguir, comprobamos que `Theta` realmente tiene señal — si estuviera cerca
de cero en la mayoría de sus entradas, la propagación colapsaría a una salida
constante para cualquier outfit, lo que produciría exactamente el síntoma que vemos
(`rank` clavado en `ln(2)`), sin que ningún `LayerNorm` pueda arreglarlo.


In [ ]:
theta_np = Theta.detach().cpu().numpy()
print(f"Theta shape: {theta_np.shape}")
print(f"min={theta_np.min():.6f}  max={theta_np.max():.6f}  mean={theta_np.mean():.6f}")
print(f"% de entradas > 1e-6 (no-cero real): {(np.abs(theta_np) > 1e-6).mean():.2%}")
print(f"Norma de Frobenius: {np.linalg.norm(theta_np):.4f}")
print(f"Suma de la diagonal (traza): {np.trace(theta_np):.4f}")
print(f"Media de los valores fuera de la diagonal: {(theta_np.sum() - np.trace(theta_np)) / (theta_np.size - theta_np.shape[0]):.6f}")

# Si esto imprime casi todo cero o una traza que domina completamente sobre el resto,
# Theta esta actuando casi como una identidad (o un cero), y la propagacion no mezcla
# informacion real entre categorias -- ese seria el bug real, no la escala/saturacion.


## 3. Dataset y DataLoader

Cada muestra es un outfit positivo + su outfit negativo correspondiente (un ítem
reemplazado por otro aleatorio del corpus, igual que en el paper). El `collate_fn` se
encarga del padding a `max_items` dentro de cada batch y de construir los tensores
finales:

- `vis`: embeddings visuales `(batch, max_items, VIS_DIM)`
- `txt`: embeddings textuales `(batch, max_items, TXT_DIM)`
- `cat`: índice de categoría de cada ítem `(batch, max_items)`
- `mask`: 1 si el ítem es real, 0 si es padding `(batch, max_items)`


In [ ]:
class OutfitDataset(Dataset):
    """Cada item del dataset es un outfit; el negativo se genera on-the-fly."""
    def __init__(self, outfits, seed=SEED):
        self.outfits = outfits
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.outfits)

    def _negative_outfit(self, items):
        neg_items    = [dict(it) for it in items]
        replace_idx  = self.rng.randint(0, len(neg_items) - 1)
        existing_ids = {it['item_id'] for it in items}
        while True:
            candidate_id = self.rng.choice(all_valid_items)
            if candidate_id not in existing_ids:
                break
        cat = item_to_category.get(candidate_id)
        neg_items[replace_idx] = {'item_id': candidate_id, 'category': cat}
        return neg_items

    def __getitem__(self, idx):
        pos_items = self.outfits[idx]['items']
        neg_items = self._negative_outfit(pos_items)
        return pos_items, neg_items


def items_to_arrays(items, max_items):
    vis  = np.zeros((max_items, VIS_DIM), dtype=np.float32)
    txt  = np.zeros((max_items, TXT_DIM), dtype=np.float32)
    cat  = np.zeros((max_items,), dtype=np.int64)
    mask = np.zeros((max_items,), dtype=np.float32)
    for i, item in enumerate(items):
        emb = get_embeddings(item['item_id'])
        if emb is None:
            continue
        vis_vec, txt_vec = emb
        vis[i]  = vis_vec
        txt[i]  = txt_vec
        cat[i]  = category_to_idx.get(item['category'], 0)
        mask[i] = 1.0
    return vis, txt, cat, mask


def collate_fn(batch):
    max_items = max(max(len(pos), len(neg)) for pos, neg in batch)

    vis_pos, txt_pos, cat_pos, mask_pos = [], [], [], []
    vis_neg, txt_neg, cat_neg, mask_neg = [], [], [], []

    for pos_items, neg_items in batch:
        v, t, c, m = items_to_arrays(pos_items, max_items)
        vis_pos.append(v); txt_pos.append(t); cat_pos.append(c); mask_pos.append(m)

        v, t, c, m = items_to_arrays(neg_items, max_items)
        vis_neg.append(v); txt_neg.append(t); cat_neg.append(c); mask_neg.append(m)

    return {
        'vis_pos':  torch.from_numpy(np.stack(vis_pos)),
        'txt_pos':  torch.from_numpy(np.stack(txt_pos)),
        'cat_pos':  torch.from_numpy(np.stack(cat_pos)).long(),
        'mask_pos': torch.from_numpy(np.stack(mask_pos)),
        'vis_neg':  torch.from_numpy(np.stack(vis_neg)),
        'txt_neg':  torch.from_numpy(np.stack(txt_neg)),
        'cat_neg':  torch.from_numpy(np.stack(cat_neg)).long(),
        'mask_neg': torch.from_numpy(np.stack(mask_neg)),
    }


train_dataset = OutfitDataset(train_outfits)
train_loader  = DataLoader(
    train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,
    collate_fn=collate_fn, num_workers=CONFIG["num_workers"],
)
print(f"Batches por epoch (train): {len(train_loader)}")

batch0 = next(iter(train_loader))
for k, v in batch0.items():
    print(f"  {k}: {tuple(v.shape)}")


## 4. Modelo

### 4.1 Capa de propagación real sobre el hipergrafo

`X' = Dropout(ReLU(Linear(Θ @ X)))`, con `Θ` fija (`(N_CATEGORIAS, N_CATEGORIAS)`) y
`X` la matriz de nodos-categoría del batch actual `(batch, N_CATEGORIAS, dim)`. Esta es
la pieza que **faltaba** en tu implementación original.


In [ ]:
class HGNNPropagation(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout_rate):
        super().__init__()
        self.linear  = nn.Linear(in_dim, hidden_dim)
        # LayerNorm ENTRE capas de propagacion: Theta es una matriz FIJA (no aprendida);
        # si su autovalor mas grande es != 1, multiplicarla repetidamente (3 capas) puede
        # disparar o aplastar la escala de las activaciones de forma consistente para
        # cualquier outfit. Esa escala descontrolada satura el sigmoid de attn_importance
        # mas adelante, matando el gradiente especificamente en la ruta de ranking (sin
        # afectar a con, que no pasa por aqui) -- sintoma: con baja, rank se queda clavado.
        self.norm    = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, X, Theta):
        # X:     (batch, n_categorias, dim_in)
        # Theta: (n_categorias, n_categorias)
        X = torch.einsum('nm,bmd->bnd', Theta, X)   # propagacion real sobre el hipergrafo
        X = self.linear(X)
        X = self.norm(X)
        X = F.relu(X)
        X = self.dropout(X)
        return X


### 4.2 Canal HGNN (un canal por modalidad)

Sigue los 3 pasos del paper (Sec. 4.3):

1. Proyección inicial de cada ítem a un espacio latente compartido `r_i = tanh(W_h f_i)`.
2. Propagación sobre el hipergrafo (`T` pasos, usando `Θ`).
3. Score de compatibilidad con atención: `x_s = Σ_i sigmoid(θ(h_i)) · leaky_relu(δ(h_i))`
   (Ec. 12).

También devolvemos `r0` (la proyección inicial, antes de propagar) para poder calcular
la pérdida de consistencia entre modalidades (Ec. 14).


In [ ]:
class HGNNChannel(nn.Module):
    def __init__(self, input_dim, n_categories, hidden_dim, n_layers, dropout_rate):
        super().__init__()
        self.n_categories = n_categories
        node_dim = hidden_dim * 2   # concat(MLP(r_i), MLP(c_i)) -> tamano 2d, fiel a la Ec. 1 del paper
                                     # ("the size of the node state is two d")

        # LayerNorm ANTES de la proyeccion: los embeddings crudos (InceptionV3 sin
        # normalizar, BoW con distinto numero de palabras activas por item) pueden tener
        # escalas muy dispares. Sin esto, Linear+tanh satura a +-1 para practicamente
        # cualquier entrada de magnitud grande, y la representacion de un item deja de
        # depender realmente de cual sea ese item -- el sintoma es una BPR loss clavada
        # en ln(2) porque score_pos y score_neg se vuelven indistinguibles.
        self.input_norm = nn.LayerNorm(input_dim)
        self.input_proj = nn.Linear(input_dim, hidden_dim)   # MLP(r_i), Ec. 1

        # c_i (Ec. 1): "a learning variable (i.e., the category features) is set for
        # each category" -- un vector APRENDIBLE por categoria, independiente del
        # embedding del item. Sin esto, el modelo no tiene ningun parametro que dependa
        # de que categoria es cada item (todo lo demas se aplica igual a cualquier
        # categoria) -- es la pieza que faltaba para poder aprender patrones de
        # compatibilidad especificos de categoria, mas alla de la co-ocurrencia fija de Theta.
        self.category_embedding = nn.Embedding(n_categories, hidden_dim)
        self.category_proj = nn.Linear(hidden_dim, hidden_dim)   # MLP(c_i), Ec. 1

        self.prop_layers = nn.ModuleList([
            HGNNPropagation(node_dim, node_dim, dropout_rate)
            for _ in range(n_layers)
        ])
        self.attn_importance = nn.Linear(node_dim, 1)
        self.attn_compat     = nn.Linear(node_dim, 1)
        # Normalizacion adicional justo antes de la atencion: es el punto exacto donde
        # una escala descontrolada tras varias pasadas de propagacion puede saturar el
        # sigmoid de attn_importance (ver comentario en HGNNPropagation).
        self.repr_norm = nn.LayerNorm(node_dim)

    def forward(self, item_embeddings, cat_idx, mask, Theta):
        mask_exp = mask.unsqueeze(-1)  # (batch, max_items, 1)

        # Paso 1: proyeccion inicial al espacio latente compartido (Ec. 1-3)
        normed = self.input_norm(item_embeddings)             # evita saturar tanh con embeddings sin escalar
        r_component = torch.tanh(self.input_proj(normed))     # MLP(r_i) -- (batch, max_items, hidden_dim)

        c_i = self.category_embedding(cat_idx)                # (batch, max_items, hidden_dim)
        c_component = torch.tanh(self.category_proj(c_i))     # MLP(c_i) -- (batch, max_items, hidden_dim)

        r0 = torch.cat([r_component, c_component], dim=-1)    # concat, Ec. 1 -- (batch, max_items, 2*hidden_dim)
        r0 = r0 * mask_exp                                     # el padding no debe aportar nada

        # Scatter: construir la matriz de nodos-categoria del hipergrafo para este batch
        one_hot = F.one_hot(cat_idx, num_classes=self.n_categories).float()  # (batch, max_items, n_cat)
        one_hot = one_hot * mask_exp
        X0 = torch.einsum('bic,bid->bcd', one_hot, r0)        # (batch, n_cat, 2*hidden_dim)

        # Paso 2: propagacion sobre el hipergrafo (T capas)
        X = X0
        for layer in self.prop_layers:
            X = layer(X, Theta)

        # Gather: recuperar la representacion actualizada de cada item via su categoria
        cat_idx_exp = cat_idx.unsqueeze(-1).expand(-1, -1, X.size(-1))   # (batch, max_items, 2*hidden_dim)
        item_repr = torch.gather(X, 1, cat_idx_exp)                     # (batch, max_items, 2*hidden_dim)
        item_repr = self.repr_norm(item_repr)                           # evita saturar el sigmoid de abajo

        # Paso 3: atencion (Ec. 8-10) -- score = sum_i sigmoid(theta(h_i)) * leaky_relu(delta(h_i))
        importance = torch.sigmoid(self.attn_importance(item_repr))                       # (batch, max_items, 1)
        compat     = F.leaky_relu(self.attn_compat(item_repr), negative_slope=0.01)       # (batch, max_items, 1)
        contrib    = importance * compat * mask_exp
        score      = contrib.squeeze(-1).sum(dim=1)                                       # (batch,)

        return score, r0


### 4.3 Modelo multimodal: fusión tardía con β + pérdida de consistencia

Dos canales independientes (visual y textual), combinados como
`x_s = β·x_vis + (1-β)·x_txt` (Ec. 13). `L_con` es la distancia L2 entre las
representaciones latentes iniciales de ambas modalidades para el mismo ítem
(Ec. 14) — únicamente sobre posiciones reales (no padding).


In [ ]:
class HGNNMultiModal(nn.Module):
    def __init__(self, vis_dim, txt_dim, n_categories, hidden_dim, n_layers, beta, dropout_rate):
        super().__init__()
        self.beta = beta
        self.vis_channel = HGNNChannel(vis_dim, n_categories, hidden_dim, n_layers, dropout_rate)
        self.txt_channel = HGNNChannel(txt_dim, n_categories, hidden_dim, n_layers, dropout_rate)

    def forward(self, vis_emb, txt_emb, cat_idx, mask, Theta):
        score_vis, r_vis = self.vis_channel(vis_emb, cat_idx, mask, Theta)
        score_txt, r_txt = self.txt_channel(txt_emb, cat_idx, mask, Theta)

        score = self.beta * score_vis + (1.0 - self.beta) * score_txt

        mask_exp = mask.unsqueeze(-1)
        l_con = ((r_vis - r_txt) * mask_exp).pow(2).sum(dim=(1, 2))   # (batch,)

        return score, l_con


model = HGNNMultiModal(
    vis_dim=VIS_DIM,
    txt_dim=TXT_DIM,
    n_categories=N_CATEGORIES,
    hidden_dim=CONFIG["hidden_dim"],
    n_layers=CONFIG["n_layers"],
    beta=CONFIG["beta"],
    dropout_rate=CONFIG["dropout_rate"],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal de parametros entrenables: {n_params:,}")


## 5. Pérdida, optimizador y paso de entrenamiento

BPR por defecto (`-log sigmoid(diff)`, estándar). Se dejan disponibles dos
alternativas: `triplet` (margen explícito) y `paper_sigmoid` — la que usa
literalmente el código oficial de OCPHN que encontraste (`-sigmoid(diff)`, sin
`log`). Su gradiente es `-σ'(diff)`, máximo en `diff=0` y que se desvanece en ambas
direcciones — un comportamiento de optimización bastante distinto a BPR estándar
(que empuja con fuerza cuando el modelo está muy equivocado).


In [ ]:
def bpr_loss(score_pos, score_neg):
    return -torch.mean(F.logsigmoid(score_pos - score_neg))

def triplet_loss(score_pos, score_neg, margin):
    return torch.mean(F.relu(margin - (score_pos - score_neg)))

def paper_sigmoid_loss(score_pos, score_neg):
    # Literal del codigo oficial de OCPHN: cost = -mean(sigmoid(s_pos - s_neg))
    return -torch.mean(torch.sigmoid(score_pos - score_neg))

def ranking_loss(score_pos, score_neg):
    if CONFIG["loss_type"] == "triplet":
        return triplet_loss(score_pos, score_neg, CONFIG["triplet_margin"])
    if CONFIG["loss_type"] == "paper_sigmoid":
        return paper_sigmoid_loss(score_pos, score_neg)
    return bpr_loss(score_pos, score_neg)


# Separamos los parametros en dos grupos: a los pesos (matrices, ndim>=2) se les aplica
# weight_decay (L2, como en el paper); a los sesgos y a los parametros de LayerNorm
# (ndim<=1) NO se les aplica -- en la version TF original la L2 solo tocaba el kernel,
# no el bias. Ademas, aplicar weight_decay a todo por igual empuja los sesgos hacia
# cero junto con los pesos, lo que facilita que el modelo colapse a una salida
# constante para minimizar L_con de forma trivial (ver nota en CONFIG["lambda_con"]).
decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if param.ndim <= 1:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.Adam([
    {"params": decay_params,    "weight_decay": CONFIG["l2_reg"]},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=CONFIG["learning_rate"])


def train_step(batch):
    model.train()
    optimizer.zero_grad()

    score_pos, lcon_pos = model(batch['vis_pos'], batch['txt_pos'], batch['cat_pos'], batch['mask_pos'], Theta)
    score_neg, lcon_neg = model(batch['vis_neg'], batch['txt_neg'], batch['cat_neg'], batch['mask_neg'], Theta)

    rank_loss = ranking_loss(score_pos, score_neg)
    con_loss  = torch.mean(lcon_pos + lcon_neg)
    loss = rank_loss + CONFIG["lambda_con"] * con_loss

    loss.backward()
    optimizer.step()

    return loss.item(), rank_loss.item(), con_loss.item()


@torch.no_grad()
def eval_step(batch):
    """Igual que train_step pero sin backward -- para medir la perdida sobre validacion."""
    model.eval()
    score_pos, lcon_pos = model(batch['vis_pos'], batch['txt_pos'], batch['cat_pos'], batch['mask_pos'], Theta)
    score_neg, lcon_neg = model(batch['vis_neg'], batch['txt_neg'], batch['cat_neg'], batch['mask_neg'], Theta)

    rank_loss = ranking_loss(score_pos, score_neg)
    con_loss  = torch.mean(lcon_pos + lcon_neg)
    loss = rank_loss + CONFIG["lambda_con"] * con_loss

    return loss.item(), rank_loss.item(), con_loss.item()


### 5.1 Diagnóstico: ¿llega gradiente a cada capa? ¿son `score_pos`/`score_neg` literalmente idénticos?

Corremos un único paso de entrenamiento "a mano" (sin `optimizer.step()`, para no
mover pesos) e inspeccionamos:
- La norma del gradiente en cada bloque del modelo — si alguna capa da `grad_norm≈0`,
  ahí está cortado el flujo.
- Los valores individuales de `score_pos` y `score_neg` del batch — si son
  literalmente idénticos (no solo parecidos), confirma que el problema es
  estructural (p.ej. `Theta` degenerada), no solo "difícil de aprender".


In [ ]:
model.train()
optimizer.zero_grad()

diag_batch = {k: v.to(device) for k, v in batch0.items()}
score_pos, lcon_pos = model(diag_batch['vis_pos'], diag_batch['txt_pos'], diag_batch['cat_pos'], diag_batch['mask_pos'], Theta)
score_neg, lcon_neg = model(diag_batch['vis_neg'], diag_batch['txt_neg'], diag_batch['cat_neg'], diag_batch['mask_neg'], Theta)

print("score_pos:", score_pos.detach().cpu().numpy())
print("score_neg:", score_neg.detach().cpu().numpy())
print("diferencia (score_pos - score_neg):", (score_pos - score_neg).detach().cpu().numpy())
print(f"¿Son EXACTAMENTE identicos?: {torch.allclose(score_pos, score_neg, atol=0.0)}")
print(f"¿Son practicamente identicos (atol=1e-4)?: {torch.allclose(score_pos, score_neg, atol=1e-4)}")

diag_loss = ranking_loss(score_pos, score_neg) + CONFIG["lambda_con"] * torch.mean(lcon_pos + lcon_neg)
diag_loss.backward()

print("\nNormas de gradiente por bloque (canal visual):")
print(f"  input_proj.weight.grad:      {model.vis_channel.input_proj.weight.grad.norm().item():.6e}")
print(f"  prop_layers[0].linear.grad:  {model.vis_channel.prop_layers[0].linear.weight.grad.norm().item():.6e}")
print(f"  prop_layers[-1].linear.grad: {model.vis_channel.prop_layers[-1].linear.weight.grad.norm().item():.6e}")
print(f"  attn_importance.weight.grad: {model.vis_channel.attn_importance.weight.grad.norm().item():.6e}")
print(f"  attn_compat.weight.grad:     {model.vis_channel.attn_compat.weight.grad.norm().item():.6e}")

optimizer.zero_grad()  # limpiamos para no contaminar el entrenamiento real


### 5.2 Prueba de sobreajuste (sanity check de arquitectura)

Si el modelo no es capaz ni de memorizar un subconjunto pequeño con muchas epochs y
un learning rate alto, hay un problema estructural real. Si sí puede, la arquitectura
funciona y lo que viste en el entrenamiento completo es cuestión de tiempo/tuning (la
señal de FITB es intrínsecamente débil, según el propio paper: *"replacing only one
part of the outfit may have little effect on the overall estimation"*).

Usamos un modelo y optimizador **independientes** (no tocan `model`/`optimizer` del
entrenamiento principal), sin dropout (queremos sobreajustar a propósito) y sin
`L_con` (para aislar si el problema es específicamente la capacidad de discriminar
ranking, no una interacción con la pérdida de consistencia).


In [ ]:
N_OVERFIT      = 200
OVERFIT_EPOCHS = 200
OVERFIT_LR     = 0.01

overfit_outfits = train_outfits[:N_OVERFIT]
overfit_dataset = OutfitDataset(overfit_outfits, seed=SEED)
overfit_loader  = DataLoader(overfit_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn, num_workers=0)

overfit_model = HGNNMultiModal(
    vis_dim=VIS_DIM, txt_dim=TXT_DIM, n_categories=N_CATEGORIES,
    hidden_dim=CONFIG["hidden_dim"], n_layers=CONFIG["n_layers"],
    beta=CONFIG["beta"], dropout_rate=0.0,   # sin dropout: queremos que pueda sobreajustar
).to(device)

overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=OVERFIT_LR)

overfit_history = []
for epoch in range(OVERFIT_EPOCHS):
    epoch_ranks = []
    for batch in overfit_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        overfit_model.train()
        overfit_optimizer.zero_grad()

        score_pos, _ = overfit_model(batch['vis_pos'], batch['txt_pos'], batch['cat_pos'], batch['mask_pos'], Theta)
        score_neg, _ = overfit_model(batch['vis_neg'], batch['txt_neg'], batch['cat_neg'], batch['mask_neg'], Theta)

        rank_loss = ranking_loss(score_pos, score_neg)   # SOLO rank, sin L_con
        rank_loss.backward()
        overfit_optimizer.step()

        epoch_ranks.append(rank_loss.item())

    overfit_history.append(np.mean(epoch_ranks))
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{OVERFIT_EPOCHS} -- rank loss: {overfit_history[-1]:.4f}")

print(f"\nrank loss inicial: {overfit_history[0]:.4f}")
print(f"rank loss final:   {overfit_history[-1]:.4f}")
print(f"ln(2) de referencia (aleatorio): {np.log(2):.4f}")

if overfit_history[-1] < 0.3:
    print("\nEl modelo SI puede sobreajustar -- la arquitectura funciona. En el dataset")
    print("completo es cuestion de tiempo/tuning, no un bug estructural.")
elif overfit_history[-1] < 0.6:
    print("\nMejora parcial -- prueba a subir OVERFIT_EPOCHS, OVERFIT_LR, o CONFIG['hidden_dim'].")
else:
    print("\nApenas mejora incluso memorizando 200 outfits -- hay algo estructural")
    print("que impide aprender, hace falta seguir investigando antes de entrenar en serio.")


## 6. Evaluación: FITB y AUC de compatibilidad

Mismo diseño que antes (evaluación outfit a outfit), adaptado al modelo de dos
canales en PyTorch, con barra de progreso.


In [ ]:
@torch.no_grad()
def score_items(items, Theta):
    """Da el score de compatibilidad para una lista de dicts {'item_id','category'}."""
    model.eval()
    n = len(items)
    vis  = np.zeros((1, n, VIS_DIM), dtype=np.float32)
    txt  = np.zeros((1, n, TXT_DIM), dtype=np.float32)
    cat  = np.zeros((1, n), dtype=np.int64)
    mask = np.ones((1, n), dtype=np.float32)

    for i, item in enumerate(items):
        emb = get_embeddings(item['item_id'])
        if emb is None:
            mask[0, i] = 0.0
            continue
        vis_vec, txt_vec = emb
        vis[0, i] = vis_vec
        txt[0, i] = txt_vec
        cat[0, i] = category_to_idx.get(item['category'], 0)

    vis_t  = torch.from_numpy(vis).to(device)
    txt_t  = torch.from_numpy(txt).to(device)
    cat_t  = torch.from_numpy(cat).to(device)
    mask_t = torch.from_numpy(mask).to(device)

    score, _ = model(vis_t, txt_t, cat_t, mask_t, Theta)
    return score.item()


def evaluate_fitb(outfits, Theta, n_candidates=4, n_samples=None, rng=None, desc="FITB"):
    rng = rng or random.Random(SEED)
    sample = outfits if n_samples is None else rng.sample(outfits, min(n_samples, len(outfits)))

    correct, total = 0, 0
    for outfit in tqdm(sample, desc=desc, leave=False):
        items = outfit['items']
        if len(items) < 2:
            continue
        mask_idx      = rng.randint(0, len(items) - 1)
        masked_item   = items[mask_idx]
        context_items = [it for i, it in enumerate(items) if i != mask_idx]
        existing_ids  = {it['item_id'] for it in items}

        candidates = {masked_item['item_id']}
        while len(candidates) < n_candidates:
            cand_id = rng.choice(all_valid_items)
            if cand_id not in existing_ids and cand_id not in candidates:
                candidates.add(cand_id)
        candidates = list(candidates)
        rng.shuffle(candidates)
        correct_idx = candidates.index(masked_item['item_id'])

        scores = []
        for cand_id in candidates:
            cand_items = context_items + [{'item_id': cand_id, 'category': item_to_category.get(cand_id)}]
            scores.append(score_items(cand_items, Theta))

        if int(np.argmax(scores)) == correct_idx:
            correct += 1
        total += 1

    return correct / total if total > 0 else 0.0


def evaluate_compatibility_auc(outfits, Theta, n_samples=None, rng=None, desc="AUC"):
    from sklearn.metrics import roc_auc_score
    rng = rng or random.Random(SEED)
    sample = outfits if n_samples is None else rng.sample(outfits, min(n_samples, len(outfits)))

    y_true, y_score = [], []
    for outfit in tqdm(sample, desc=desc, leave=False):
        items = outfit['items']
        score_pos = score_items(items, Theta)

        neg_items    = [dict(it) for it in items]
        replace_idx  = rng.randint(0, len(neg_items) - 1)
        existing_ids = {it['item_id'] for it in items}
        while True:
            cand_id = rng.choice(all_valid_items)
            if cand_id not in existing_ids:
                break
        neg_items[replace_idx] = {'item_id': cand_id, 'category': item_to_category.get(cand_id)}
        score_neg = score_items(neg_items, Theta)

        y_true.extend([1, 0])
        y_score.extend([score_pos, score_neg])

    return roc_auc_score(y_true, y_score)


## 7. Checkpointing + early stopping

Se guarda en `checkpoints_HGNN/best_model.pt` el `state_dict` del modelo con mejor
AUC de validación visto hasta el momento; el entrenamiento se detiene si no hay
mejora durante `CONFIG["early_stopping_patience"]` epochs consecutivas.


In [ ]:
class BestModelCheckpoint:
    def __init__(self, save_path, verbose=True):
        self.save_path  = save_path
        self.verbose    = verbose
        self.best_score = -np.inf
        self.best_epoch = 0
        os.makedirs(save_path, exist_ok=True)
        self.ckpt_file = os.path.join(save_path, "best_model.pt")

    def on_epoch_end(self, epoch, model, auc):
        if auc > self.best_score:
            self.best_score = auc
            self.best_epoch = epoch + 1
            torch.save(model.state_dict(), self.ckpt_file)
            if self.verbose:
                print(f"  -> Mejor modelo guardado (AUC val: {auc:.4f})")
        elif self.verbose:
            print(f"  -> Sin mejora (mejor AUC val: {self.best_score:.4f} en epoch {self.best_epoch})")

    def restore_best(self, model):
        model.load_state_dict(torch.load(self.ckpt_file, map_location=device))
        print(f"Modelo restaurado -- mejor epoch: {self.best_epoch}, AUC val: {self.best_score:.4f}")


class EarlyStopping:
    def __init__(self, patience=5, verbose=True):
        self.patience    = patience
        self.verbose     = verbose
        self.best_score  = -np.inf
        self.counter     = 0
        self.should_stop = False

    def on_epoch_end(self, auc):
        if auc > self.best_score:
            self.best_score = auc
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"  -> Early stopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.should_stop = True
                print("  -> Early stopping activado")


## 8. Loop de entrenamiento

Barra de progreso con `tqdm` dentro de cada epoch, mostrando la pérdida acumulada
(`loss`, `rank`, `con`) en tiempo real.


In [ ]:
checkpoint    = BestModelCheckpoint(PATHS["checkpoints"])
early_stopper = EarlyStopping(patience=CONFIG["early_stopping_patience"])
eval_rng      = random.Random(SEED + 1)

# DataLoader de validacion (sin shuffle, mismo collate_fn que en train) -- necesario
# para poder medir Loss_val por epoch, no solo AUC_val/FITB_val.
valid_dataset = OutfitDataset(valid_outfits, seed=SEED + 2)
valid_loader  = DataLoader(
    valid_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    collate_fn=collate_fn, num_workers=CONFIG["num_workers"],
)

history = []

for epoch in range(CONFIG["max_epochs"]):
    epoch_losses, epoch_rank, epoch_con = [], [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['max_epochs']}", leave=True)
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss, rank_loss, con_loss = train_step(batch)

        epoch_losses.append(loss)
        epoch_rank.append(rank_loss)
        epoch_con.append(con_loss)

        pbar.set_postfix({
            "loss": f"{np.mean(epoch_losses):.4f}",
            "rank": f"{np.mean(epoch_rank):.4f}",
            "con":  f"{np.mean(epoch_con):.4f}",
        })

    # --- Perdida de validacion (forward puro, sin backward -- barato) ---
    epoch_val_rank, epoch_val_con = [], []
    for vbatch in tqdm(valid_loader, desc=f"Val loss {epoch+1}/{CONFIG['max_epochs']}", leave=False):
        vbatch = {k: v.to(device) for k, v in vbatch.items()}
        _, v_rank, v_con = eval_step(vbatch)
        epoch_val_rank.append(v_rank)
        epoch_val_con.append(v_con)

    fitb_val  = evaluate_fitb(valid_outfits, Theta, n_samples=CONFIG["eval_n_samples_per_epoch"], rng=eval_rng, desc="FITB (val)")
    auc_val   = evaluate_compatibility_auc(valid_outfits, Theta, n_samples=CONFIG["eval_n_samples_per_epoch"], rng=eval_rng, desc="AUC (val)")
    # AUC sobre una muestra de TRAIN del mismo tamano que la de val, para que ambas curvas
    # sean comparables (no se usa para seleccionar el modelo, solo para la figura).
    auc_train = evaluate_compatibility_auc(train_outfits, Theta, n_samples=CONFIG["eval_n_samples_per_epoch"], rng=eval_rng, desc="AUC (train)")

    print(f"Epoch {epoch+1:02d}/{CONFIG['max_epochs']} "
          f"-- loss: {np.mean(epoch_losses):.4f} (rank: {np.mean(epoch_rank):.4f}, con: {np.mean(epoch_con):.4f}) "
          f"-- val_loss: rank {np.mean(epoch_val_rank):.4f}, con {np.mean(epoch_val_con):.4f} "
          f"-- FITB(val): {fitb_val:.4f} -- AUC(train): {auc_train:.4f} -- AUC(val): {auc_val:.4f}")

    history.append({
        "epoch": epoch + 1,
        "loss": float(np.mean(epoch_losses)),
        "rank_loss": float(np.mean(epoch_rank)), "con_loss": float(np.mean(epoch_con)),
        "val_rank_loss": float(np.mean(epoch_val_rank)), "val_con_loss": float(np.mean(epoch_val_con)),
        "fitb_val": fitb_val, "auc_train": auc_train, "auc_val": auc_val,
    })

    checkpoint.on_epoch_end(epoch, model, auc_val)
    early_stopper.on_epoch_end(auc_val)

    if early_stopper.should_stop:
        print(f"\nEntrenamiento detenido en epoch {epoch+1}")
        break

checkpoint.restore_best(model)


### 8.1 Curvas de aprendizaje (figuras para la memoria)

Se generan dos figuras independientes -- pensadas para combinarse en LaTeX con `subcaption` como paneles (a) y (b) de una misma `figure` -- a partir de `history`:

- **(a)** $\mathcal{L}_{\mathrm{BPR}}$ de entrenamiento (línea continua) frente a validación (línea discontinua).
- **(b)** AUC de compatibilidad en entrenamiento (línea continua) frente a validación (línea discontinua), con la época óptima marcada.

En ambas se señala con una línea vertical punteada la época en la que se activó el *early stopping*. El término de consistencia $\mathcal{L}_{\text{con}}$ no se incluye aquí (tiene una escala y un propósito distintos -- ver figura de la sección 7.3.2); se genera aparte en la celda siguiente.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# --- Configuración global de estilos para TFM ---
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "axes.edgecolor": "0.3",
    "figure.dpi": 200,
})

# --- Extracción de datos ---
epochs        = [h["epoch"]         for h in history]
rank_train    = [h["rank_loss"]     for h in history]
rank_val      = [h["val_rank_loss"] for h in history]
auc_train_h   = [h["auc_train"]     for h in history]
auc_val_h     = [h["auc_val"]       for h in history]

best_epoch = checkpoint.best_epoch     # Epoch del mejor AUC val
best_auc   = checkpoint.best_score
stop_epoch = epochs[-1]                # Época de activación del early stopping

# Colores institucionales / académicos
COLOR_TRAIN = "#1f77b4"  # Azul para entrenamiento
COLOR_VAL   = "#ff7f0e"  # Naranja para validación
COLOR_OPT   = "#2ca02c"  # Verde para la marca del óptimo

# Función auxiliar para marcar Early Stopping
def mark_early_stopping(ax, y_top_frac=0.95):
    ax.axvline(stop_epoch, color="0.5", lw=1.1, ls="--")
    ylo, yhi = ax.get_ylim()
    ax.text(stop_epoch - 0.15, ylo + (yhi - ylo) * y_top_frac,
            "Early stopping", fontsize=8.0, color="0.35",
            rotation=90, ha="right", va="top")

# =========================================================
# Crear Figura Unificada con 2 Paneles (a) y (b)
# =========================================================
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(10.5, 4.0))

# ---------------------------------------------------------
# Panel (a): Pérdida BPR (Train vs Val)
# ---------------------------------------------------------
ax_a.plot(epochs, rank_train, color=COLOR_TRAIN, ls="-", marker="o", ms=4, lw=1.5,
          label=r"$\mathcal{L}_{\mathrm{BPR}}^{\mathrm{train}}$")
ax_a.plot(epochs, rank_val, color=COLOR_VAL, ls="--", marker="s", ms=4, lw=1.5,
          label=r"$\mathcal{L}_{\mathrm{BPR}}^{\mathrm{val}}$")

ax_a.set_xlabel("Época")
ax_a.set_ylabel(r"Pérdida BPR, $\mathcal{L}_{\mathrm{BPR}}$")
ax_a.set_xticks(epochs)
ax_a.set_xlim(epochs[0] - 0.45, epochs[-1] + 0.8)
mark_early_stopping(ax_a, y_top_frac=0.95)
ax_a.legend(loc="upper right", frameon=True, edgecolor="none", facecolor="white", framealpha=0.8)
ax_a.set_title("(a) Evolución de la pérdida BPR")

# ---------------------------------------------------------
# Panel (b): Rendimiento AUC (Train vs Val)
# ---------------------------------------------------------
ax_b.plot(epochs, auc_train_h, color=COLOR_TRAIN, ls="-", marker="o", ms=4, lw=1.5,
          label=r"AUC$^{\mathrm{train}}$")
ax_b.plot(epochs, auc_val_h, color=COLOR_VAL, ls="--", marker="s", ms=4, lw=1.5,
          label=r"AUC$^{\mathrm{val}}$")

# Marca del óptimo alcanzado en validación
ax_b.scatter([best_epoch], [best_auc], color=COLOR_OPT, s=110, zorder=5,
             edgecolor="black", linewidth=0.8, marker="*")
ax_b.annotate(f"Óptimo: Época {best_epoch}\nAUC = {best_auc:.4f}",
              xy=(best_epoch, best_auc), xytext=(best_epoch - 3.5, best_auc - 0.12),
              fontsize=8.5, color="black",
              arrowprops=dict(arrowstyle="->", color="black", lw=0.9))

ax_b.set_xlabel("Época")
ax_b.set_ylabel("AUC de compatibilidad")
ax_b.set_xticks(epochs)
ax_b.set_xlim(epochs[0] - 0.45, epochs[-1] + 0.8)
ax_b.set_ylim(0.45, 0.75)  # Ajustado al rango real de trabajo para mayor legibilidad
ax_b.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
mark_early_stopping(ax_b, y_top_frac=0.95)
ax_b.legend(loc="lower right", frameon=True, edgecolor="none", facecolor="white", framealpha=0.8)
ax_b.set_title("(b) Rendimiento de validación (AUC)")

# Guardar figura unificada
plt.tight_layout()
plt.savefig("./figuras/HGNN_entrenamiento.pdf", bbox_inches="tight")
plt.savefig("./figuras/HGNN_entrenamiento.png", bbox_inches="tight", dpi=300)
plt.show()

## 9. Evaluación final en test

In [ ]:
print("Evaluando en el conjunto de test completo (puede tardar varios minutos)...")

fitb_test = evaluate_fitb(test_outfits, Theta, desc="FITB (test)")
auc_test  = evaluate_compatibility_auc(test_outfits, Theta, desc="AUC (test)")

print("\n" + "="*50)
print("RESULTADOS FINALES (TEST)")
print("="*50)
print(f"FITB accuracy: {fitb_test:.4f}   (baseline aleatorio: 0.25)")
print(f"AUC compat.:   {auc_test:.4f}   (baseline aleatorio: 0.50)")


### 9.1 Guardar historial y configuración usados

In [ ]:
run_summary = {
    "config": CONFIG,
    "n_categories": N_CATEGORIES,
    "n_train_outfits": len(train_outfits),
    "n_valid_outfits": len(valid_outfits),
    "n_test_outfits": len(test_outfits),
    "history": history,
    "test_fitb_accuracy": fitb_test,
    "test_auc": auc_test,
}

with open(os.path.join(PATHS["checkpoints"], "run_summary.json"), "w") as f:
    json.dump(run_summary, f, indent=2)

print(f"Resumen guardado en: {os.path.join(PATHS['checkpoints'], 'run_summary.json')}")


## 10. Notas finales / próximos pasos

- Si `hidden_dim=12` (valor óptimo reportado por el paper) rinde peor de lo esperado
  con estos embeddings de mayor dimensión (2048 visual / ~2900 BoW) que las del paper
  original, prueba a subirlo (64–128) — es la desviación más probable entre "fiel al
  paper" y "lo que realmente funciona bien" con tus features concretas.
- `CONFIG["lambda_con"]` no tiene un valor de referencia en el paper; si ves que el
  entrenamiento prioriza demasiado igualar visual/textual en detrimento del ranking,
  bájalo.
- `CONFIG["num_workers"]` está en 0 por defecto para evitar problemas de pickling con
  multiprocessing en Windows. En Linux puedes subirlo (p.ej. 4) para acelerar la carga
  de batches en paralelo.
- Si quieres comparar contra tu implementación anterior (embeddings CLIP, un único
  canal), puedes mantener ese notebook aparte — no es necesario tirarlo, ahora tienes
  ambos enfoques para comparar resultados.
